<a href="https://colab.research.google.com/github/brandonwolk/Kaneohe-Bay-Coral-Restoration-GIS/blob/main/capstone/01_Data_Ingestion_and_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# MODULE 1: DATA INGESTION, BATHYMETRY PROCESSING, AND ERDDAP HARVESTING
# Kāneʻohe Bay Coral Restoration Spatial Framework
# ==============================================================================

import os
import calendar
import time
import requests
import rasterio
from rasterio.fill import fillnodata
import geopandas as gpd
import numpy as np
from shapely.geometry import box
from google.colab import drive

# 1. Mount Google Drive and Set Directories
drive.mount('/content/drive', force_remount=True)
CAPSTONE_ROOT = "/content/drive/MyDrive/Capstone"
RAW_DATA_DIR = os.path.join(CAPSTONE_ROOT, "01 Raw Data")
PROCESSED_DIR = os.path.join(CAPSTONE_ROOT, "02 Processed Rasters")
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

BBOX_WGS84 = (-157.86, 21.40, -157.73, 21.54)

# 2. Bathymetry Processing (4m LiDAR Standard -> EPSG:6634)
def process_bathymetry():
    in_path = os.path.join(RAW_DATA_DIR, 'kaneohe_MB_Lidar_4m.asc')
    out_path = os.path.join(PROCESSED_DIR, 'Kaneohe_Bathy_Final.tif')

    with rasterio.open(in_path) as src:
        data = src.read(1)
        data_to_fill = np.where((data > -100) & (data < 2), data, -9999)
        filled = fillnodata(data_to_fill, mask=(data_to_fill != -9999), max_search_distance=10)

        profile = src.profile.copy()
        profile.update(driver='GTiff', nodata=-9999, crs='EPSG:6634')
        for key in ['BLOCKXSIZE', 'BLOCKYSIZE', 'TILED']:
            profile.pop(key, None)

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(filled, 1)
    print(f"Bathymetry processed and saved to: {out_path}")

# 3. Benthic Habitat Vector Clipping
def process_benthic():
    in_path = os.path.join(RAW_DATA_DIR, 'hi_noaa_oahu_benthic_habitats.zip')
    shp_out_path = os.path.join(PROCESSED_DIR, 'Kaneohe_Benthic.shp')

    gdf = gpd.read_file(f"zip://{in_path}")
    bbox_gdf = gpd.GeoDataFrame(geometry=[box(*BBOX_WGS84)], crs="EPSG:4326")

    gdf_proj = gdf.to_crs("EPSG:6634")
    bbox_proj = bbox_gdf.to_crs("EPSG:6634")
    clipped = gpd.clip(gdf_proj, bbox_proj)
    clipped.to_file(shp_out_path, driver='ESRI Shapefile')
    print(f"Benthic shapefile created at: {shp_out_path}")

if __name__ == "__main__":
    process_bathymetry()
    process_benthic()

Mounted at /content/drive


Bathymetry processed and saved to: /content/drive/MyDrive/Capstone/02 Processed Rasters/Kaneohe_Bathy_Final.tif
Benthic shapefile created at: /content/drive/MyDrive/Capstone/02 Processed Rasters/Kaneohe_Benthic.shp
